In [0]:
-- ╔════════════════════════════════════════════════════════════╗
-- ║  02_Checkin_Silver_Cleaned                                ║
-- ║  Silver_Cleaned Checkin Tables — Empty DDL Only           ║
-- ╚════════════════════════════════════════════════════════════╝
--
-- Source : yelp_poc_uc.bronze_final.yelp_checkin (131,930 rows)
-- Targets (all in yelp_poc_uc.silver_cleaned):
--   yelp_checkin  (1:N per business; aggregated with checkin_count)
--
-- ════════════════════════════════════════════════════════════
-- RULES & DESIGN PRINCIPLES
-- ════════════════════════════════════════════════════════════
-- (Complete definitions in 01_Business_Silver_Cleaned)
--
-- R1  DDL-Only / No Population
--     Tables are created empty. Data population (split → explode
--     → to_timestamp → aggregate COUNT) is handled by a
--     separate loading script. This keeps DDL idempotent and
--     safe to re-run without side effects on existing data.
--
-- R2  No Unnecessary Constraint Removal
--     PK/FK/NOT NULL declared inline in CREATE TABLE.
--     CREATE OR REPLACE TABLE resets the whole definition
--     atomically, so separate ALTER TABLE is unnecessary.
--
-- R3  Normalization of Nested Data
--     The bronze source stores all checkin dates for a business
--     as a single comma-separated STRING column. Silver Cleaned
--     normalizes this to one row per checkin event:
--       • Bronze: 1 row per business, CSV dates column
--       • Silver: 1 row per (business_id, checkin_timestamp)
--
-- R4  Referential Integrity via PK/FK
--     Composite PK on (business_id, checkin_timestamp) and FK
--     referencing the parent yelp_business(business_id) are
--     declared inline. These are informational in Databricks
--     but document the intended relationship.
--
-- R5  Semantic Data Types
--     Column types reflect real-world meaning, not bronze storage:
--
--     Bronze Type        →  Silver Semantic Type  → Columns Affected
--     ────────────────────────────────────────────────────────────
--     STRING (CSV dates) →  TIMESTAMP             → checkin_timestamp
--     (derived)          →  BIGINT                → checkin_count (COUNT of duplicate rows per key)
--
-- R6  No Ingestion Timestamps
--     No _ingestion_timestamp columns are added. Lineage and
--     provenance are tracked via TBLPROPERTIES metadata.
--
-- R7  Self-Documenting DDL
--     Every column carries an inline comment describing its
--     business meaning, source mapping, and type transformation.
--     Table-level COMMENTs summarize purpose and cardinality.
--
-- DQ Issues Addressed (during population):
--   DQ-CHECKIN-001: CSV dates packed in single column → exploded
--   DQ-CHECKIN-002: Dates stored as strings → cast to TIMESTAMP
--   DQ-CHECKIN-005: Duplicate timestamps (~3,543 excess) → aggregated
--                   into checkin_count instead of quarantined
--
-- Execution order: Run AFTER 01_Business (FK dependency).
-- ════════════════════════════════════════════════════════════


-- ── 1. Checkin Table (Silver Cleaned) ───────────────────────
-- Bronze path: yelp_checkin
--                └── .date  (STRING, comma-separated timestamps)
-- Individual checkin events normalized from the bronze CSV-
-- packed date column. One row per (business_id, checkin_timestamp).
-- Duplicates aggregated into checkin_count during population.

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_checkin (
  business_id        STRING     NOT NULL  COMMENT 'FK to yelp_business. Unique 22-char Yelp identifier for the business. Part of composite PK with checkin_timestamp.',
  checkin_timestamp  TIMESTAMP  NOT NULL  COMMENT 'Individual checkin event timestamp. Source: bronze_final.yelp_checkin.date CSV string, split and exploded into individual rows, then cast from STRING to TIMESTAMP.',
  checkin_count      BIGINT     NOT NULL  COMMENT 'Number of checkins recorded at this exact (business_id, checkin_timestamp) pair. Derived during population as COUNT(*) of duplicate rows per composite key. Most rows have count=1; ~3,543 excess rows across ~3,499 key groups were aggregated rather than quarantined (DQ-CHECKIN-005).',
  CONSTRAINT pk_sc_checkin PRIMARY KEY (business_id, checkin_timestamp),
  CONSTRAINT fk_sc_checkin_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Silver_Cleaned — normalized checkin events (DDL-only, no data). Each row represents a unique (business_id, checkin_timestamp) pair with a checkin_count recording how many checkins occurred at that exact moment. Normalized from bronze_final.yelp_checkin (131,930 rows, one per business with a single comma-separated date string) via: split → explode → cast to TIMESTAMP → GROUP BY with COUNT. Composite PK on (business_id, checkin_timestamp). FK to silver_cleaned.yelp_business(business_id). DQ issues resolved: DQ-CHECKIN-001 (CSV packed dates), DQ-CHECKIN-002 (string dates), DQ-CHECKIN-005 (duplicate timestamps aggregated into checkin_count).'
TBLPROPERTIES (
  'quality'                = 'silver_cleaned',
  'pipelines.sourceTable'  = 'yelp_poc_uc.bronze_final.yelp_checkin',
  'design.populationMode'  = 'ddl_only',
  'design.normalizedFrom'  = 'yelp_poc_uc.bronze_final.yelp_checkin',
  'dq.issues.resolved'     = 'DQ-CHECKIN-001, DQ-CHECKIN-002, DQ-CHECKIN-005'
);


-- ── 2. Verification ─────────────────────────────────────────

SELECT 'yelp_checkin' AS table_name, 'silver_cleaned' AS layer, COUNT(*) AS total_rows
FROM yelp_poc_uc.silver_cleaned.yelp_checkin;